# Loan Support Agent

This project implements an autonomous analytical agent capable of:
- profiling structured data,
- training a baseline classification model,
- evaluating feature importance,
- generating automated decision-support insights

## 1. Problem Definition

The objective is to develop an analytical agent capable of supporting loan approval decisions by identifying key drivers of credit outcomes and generating structured insights

## 2. Data Overview


In [159]:
import pandas as pd

DATA_PATH = "loan_data.csv"
TARGET_COL = "loan_status"

df = pd.read_csv(DATA_PATH)

n_rows, n_cols = df.shape

print(f"Number of observations: {n_rows}")
print(f"Number of features: {n_cols - 1}")
print(f"Target column: {TARGET_COL}")

print("\nPreview of the dataset:")
display(df.head(5))

print("\nTarget distribution (counts):")
print(df[TARGET_COL].value_counts())

print("\nTarget distribution (proportion):")
print(df[TARGET_COL].value_counts(normalize=True).round(4))



Number of observations: 45000
Number of features: 13
Target column: loan_status

Preview of the dataset:


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1



Target distribution (counts):
loan_status
0    35000
1    10000
Name: count, dtype: int64

Target distribution (proportion):
loan_status
0    0.7778
1    0.2222
Name: proportion, dtype: float64


### Observations

- The dataset contains 45,000 observations and 13 predictive features
- The target variable `loan_status` is binary
- The class distribution is moderately imbalanced (~78% vs ~22%)

## 3. Data Profiling

Before proceeding to modeling, the agent performs a structural quality assessment of the dataset, including:

- missing value detection,
- duplicate row detection,
- feature type identification,
- categorical and numerical feature separation


In [160]:
# Missing values
total_missing = df.isnull().sum().sum()

# Duplicate rows
duplicate_rows = df.duplicated().sum()

# Feature types
categorical_features = df.select_dtypes(include=["object"]).columns.tolist()
numerical_features = df.select_dtypes(exclude=["object"]).columns.tolist()

print("Total missing values:", total_missing)
print("Duplicate rows:", duplicate_rows)

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print([col for col in numerical_features if col != TARGET_COL])

Total missing values: 0
Duplicate rows: 0

Categorical features:
['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']

Numerical features:
['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']


### Profiling Summary

- The dataset contains no missing values and no duplicate records
- Several categorical features are present and will require encoding
- Numerical features are suitable for baseline modeling without scaling at this stage

The dataset is structurally ready for preprocessing and model training

## 4. Preprocessing Strategy

The agent prepares the dataset for modeling by:

- separating features and target,
- encoding categorical variables,
- scaling numerical variables,
- splitting data into training and test sets

In [161]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop(TARGET_COL, axis=1)
y = df[TARGET_COL]

categorical_features = ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']
numerical_features = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
                      'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
                      'credit_score']

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (36000, 13)
Test set size: (9000, 13)


### Preprocessing Summary

The dataset was split into training (80%) and test (20%) subsets.

Given the moderate class imbalance, stratified sampling was applied to preserve class distribution across splits.
The dataset size (45,000 observations) allows reliable model training and evaluation

## 5. Baseline Modeling

At this stage, the agent trains a baseline classification model.

Logistic Regression was selected because:

- The target variable (`loan_status`) is binary
- The model provides probabilistic outputs suitable for risk-based decision systems
- It is interpretable and appropriate as a reference baseline

### Evaluation Strategy

The dataset exhibits moderate class imbalance (~78% vs ~22%).  
In such settings, accuracy alone can be misleading.

For example, a naive model predicting only the majority class would achieve ~78% accuracy while having no discriminatory power.

Therefore, the agent evaluates performance using:

- **Precision** — reliability of positive predictions
- **Recall** — ability to detect positive cases
- **F1-score** — balance between precision and recall
- **ROC-AUC** — the model’s ability to rank positive cases higher than negative ones across all classification thresholds

ROC-AUC is particularly relevant in decision-support and credit risk contexts, as it measures the quality of probabilistic ranking rather than threshold-based classification

In [162]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Full modeling pipeline: preprocessing + classifier
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train
model.fit(X_train, y_train)

# Predict class labels and probabilities
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Metrics (useful under class imbalance)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

print("Baseline model: Logistic Regression\n")
for k, v in metrics.items():
    print(f"{k:10s}: {v:.4f}")

print("\nConfusion matrix [ [TN FP], [FN TP] ]:")
print(confusion_matrix(y_test, y_pred))

Baseline model: Logistic Regression

accuracy  : 0.8993
precision : 0.7888
recall    : 0.7470
f1        : 0.7673
roc_auc   : 0.9562

Confusion matrix [ [TN FP], [FN TP] ]:
[[6600  400]
 [ 506 1494]]


### Model Performance Interpretation

The baseline logistic regression model demonstrates strong predictive performance.

- Accuracy (~90%) indicates high overall correctness
- Precision (~0.79) suggests that positive predictions are reliable
- Recall (~0.75) shows that the model successfully identifies most positive cases
- F1-score (~0.77) reflects a balanced trade-off between precision and recall
- ROC-AUC (~0.96) indicates excellent discriminatory power and strong ranking capability

The confusion matrix suggests that the model maintains a reasonable balance between false positives and false negatives, making it suitable for decision-support applications

The model provides a stable baseline for further interpretability analysis

## 6. Model Interpretability

A decision-support agent must not only predict outcomes, but also explain the reasoning behind its decisions.

Interpretability is particularly important in financial contexts, where transparency and accountability are required.

To understand which factors influence loan approval decisions, the agent analyzes the coefficients of the logistic regression model and ranks features by their impact.

### Methodological Note: Working with Pipelines and One-Hot Encoding

The preprocessing stage applies different transformations to numerical and categorical features using a `ColumnTransformer`.  
Categorical variables are expanded via One-Hot Encoding, which increases the dimensionality of the feature space.

As a result, the feature representation used by the model differs from the original dataset structure.

For example, a single categorical column such as `loan_intent` may be transformed into multiple binary features (e.g., `loan_intent_personal`, `loan_intent_education`, etc.).

Therefore, when interpreting model coefficients, it is essential to reconstruct the transformed feature names from the preprocessing pipeline.  
Failing to do so would lead to incorrect mapping between coefficients and their corresponding features.

Careful alignment between transformed features and model parameters ensures interpretability, transparency, and methodological correctness.

In [163]:
import numpy as np
import pandas as pd

# Get feature names after preprocessing
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

# Get transformed feature names
num_features = numerical_features
cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)

all_features = np.concatenate([num_features, cat_features])

# Get coefficients
coefficients = classifier.coef_[0]

feature_importance = pd.DataFrame({
    "feature": all_features,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
})

# Sort by absolute impact
feature_importance = feature_importance.sort_values(by="abs_coefficient", ascending=False)

feature_importance.head(10)

,feature,coefficient,abs_coefficient
26,previous_loan_defaults_on_file_Yes,-5.072659,5.072659
25,previous_loan_defaults_on_file_No,3.136223,3.136223
17,person_home_ownership_OWN,-1.818956,1.818956
5,loan_percent_income,1.348589,1.348589
24,loan_intent_VENTURE,-0.998544,0.998544
4,loan_int_rate,0.990910,0.990910
8,person_gender_female,-0.977714,0.977714
9,person_gender_male,-0.958722,0.958722
20,loan_intent_EDUCATION,-0.702571,0.702571
3,loan_amnt,-0.611590,0.611590


### Feature Importance Interpretation

The logistic regression coefficients allow separation of influential factors into negative and positive contributors to loan approval probability.

#### Strong Negative Factors (reduce approval probability)

- previous_loan_defaults_on_file_Yes (−5.07)
- person_home_ownership_OWN (−1.82)
- loan_intent_VENTURE (−0.99)
- person_gender_female (−0.98)
- person_gender_male (−0.96)
- loan_intent_EDUCATION (−0.70)
- loan_amnt (−0.61)

The strongest negative factor is prior default history, which substantially decreases approval likelihood. Larger loan amounts also reduce approval probability. Certain loan purposes and home ownership status demonstrate moderate negative effects.

#### Strong Positive Factors (increase approval probability)

- previous_loan_defaults_on_file_No (+3.14)
- loan_percent_income (+1.35)
- loan_int_rate (+0.99)

The absence of prior defaults significantly increases approval probability. Higher loan-to-income ratios and higher interest rates also positively correlate with approval in this dataset, potentially reflecting internal risk-pricing mechanisms.

---

### Fairness and Bias Considerations

Gender-related features (person_gender_female: −0.98, person_gender_male: −0.96) appear among the top influential predictors.

In real-world credit systems, the presence of gender as a strong predictive factor raises fairness and ethical concerns. Financial decision systems are typically required to comply with non-discrimination principles.

If gender meaningfully influences model predictions, this may lead to:
- disparate approval rates across demographic groups,
- indirect discrimination,
- regulatory and compliance risks,
- reduced trust in automated decision systems.

---

### Recommended Fairness Assessment

To address this issue, additional validation steps are required:

1. Group-wise performance evaluation  
   Compare approval rates, precision, recall, and false positive/negative rates across gender groups

2. Counterfactual testing  
   Simulate predictions while holding all other features constant and changing only gender to evaluate direct model sensitivity

3. Feature removal experiment  
   Retrain the model without gender features and compare performance metrics.  
   If performance remains stable, removing gender may reduce bias without harming predictive quality

4. Fairness metrics  
   Evaluate statistical parity difference or equal opportunity difference between groups

---

### Mitigation Strategies

If gender influence is deemed problematic:

- Remove gender from the feature set
- Apply fairness-constrained optimization methods
- Introduce post-processing threshold adjustments to equalize group performance

Responsible AI deployment requires balancing predictive performance (ROC-AUC ≈ 0.96) with fairness, transparency, and regulatory compliance.

## 6.1 Group-wise Performance Evaluation (Gender)

A responsible decision-support agent should be evaluated not only on overall metrics, but also across demographic groups.

Here, the agent compares model behavior across gender groups by reporting:
- **approval rate** (true positive share),
- **predicted approval rate** (predicted positive rate),
- classification metrics (precision, recall, F1, ROC-AUC),
- and error rates (TPR/FPR), derived from the confusion matrix

This helps detect potential disparities between groups.

In [164]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

SENSITIVE_COL = "person_gender"

# Predictions from the already-trained baseline model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

audit_df = X_test.copy()
audit_df["y_true"] = y_test.values
audit_df["y_pred"] = y_pred
audit_df["y_proba"] = y_proba

def compute_group_report(df, group_col):
    reports = []
    for g in sorted(df[group_col].dropna().unique()):
        d = df[df[group_col] == g]
        if len(d) == 0:
            continue

        # base rates
        true_pos_share = (d["y_true"] == 1).mean()          # actual approval share
        pred_pos_rate = (d["y_pred"] == 1).mean()           # predicted approval rate

        # metrics
        acc = accuracy_score(d["y_true"], d["y_pred"])
        prec = precision_score(d["y_true"], d["y_pred"], zero_division=0)
        rec = recall_score(d["y_true"], d["y_pred"], zero_division=0)
        f1 = f1_score(d["y_true"], d["y_pred"], zero_division=0)

        auc = roc_auc_score(d["y_true"], d["y_proba"]) if d["y_true"].nunique() > 1 else np.nan

        # confusion-derived rates
        cm = confusion_matrix(d["y_true"], d["y_pred"], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        tpr = tp / (tp + fn) if (tp + fn) else np.nan  # True Positive Rate (recall)
        fpr = fp / (fp + tn) if (fp + tn) else np.nan  # False Positive Rate

        reports.append({
            "group": g,
            "n": len(d),
            "true_positive_share": round(true_pos_share, 4),
            "predicted_positive_rate": round(pred_pos_rate, 4),
            "accuracy": round(acc, 4),
            "precision": round(prec, 4),
            "recall": round(rec, 4),
            "f1": round(f1, 4),
            "roc_auc": round(auc, 4) if not np.isnan(auc) else np.nan,
            "TPR": round(tpr, 4) if not np.isnan(tpr) else np.nan,
            "FPR": round(fpr, 4) if not np.isnan(fpr) else np.nan,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
        })

    return pd.DataFrame(reports)

group_report = compute_group_report(audit_df, SENSITIVE_COL)
group_report

,group,n,true_positive_share,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,TPR,FPR,TN,FP,FN,TP
0,female,3945,0.2165,0.2112,0.8923,0.7575,0.7389,0.7481,0.9522,0.7389,0.0654,2889,202,223,631
1,male,5055,0.2267,0.2099,0.9048,0.8134,0.7531,0.7821,0.9594,0.7531,0.0507,3711,198,283,863


The predicted approval rates are nearly identical across gender groups (~21%), with a difference of approximately 0.0013.

Such a minimal gap suggests that the model does not introduce or amplify group-level disparities in approval outcomes.

## 6.2 Counterfactual Sensitivity Test (Flip Gender)

To test whether gender directly influences predictions, the agent performs a counterfactual experiment:
- keep all features unchanged,
- modify **only** the gender feature (male ↔ female),
- compare predicted probabilities and decisions.

If changing only gender frequently changes the output, the model may be directly sensitive to this attribute.

In [165]:
# Create counterfactual copy: flip gender only
X_cf = X_test.copy()
X_cf[SENSITIVE_COL] = X_cf[SENSITIVE_COL].map({"male": "female", "female": "male"}).fillna(X_test[SENSITIVE_COL])

proba_orig = model.predict_proba(X_test)[:, 1]
proba_cf = model.predict_proba(X_cf)[:, 1]

delta = proba_cf - proba_orig

# Decision changes at threshold 0.5 (baseline threshold)
pred_orig = (proba_orig >= 0.5).astype(int)
pred_cf = (proba_cf >= 0.5).astype(int)

share_changed = (pred_cf != pred_orig).mean()

print("Counterfactual test (flip gender only):")
print(f"Mean Δprobability:   {delta.mean():.4f}")
print(f"Median Δprobability: {np.median(delta):.4f}")
print(f"Share of changed decisions (thr=0.5): {share_changed:.4f}")

Counterfactual test (flip gender only):
Mean Δprobability:   -0.0002
Median Δprobability: -0.0000
Share of changed decisions (thr=0.5): 0.0017


### Counterfactual Sensitivity Interpretation

The counterfactual experiment reveals minimal direct sensitivity of the model to gender.

The mean change in predicted probability after flipping gender is approximately −0.0002, and the median change is effectively zero. This indicates that, on average, modifying only the gender attribute has a negligible impact on predicted risk scores.

Furthermore, only 0.17% of decisions change when gender is flipped while all other features remain constant.

These results suggest that although gender appears among influential coefficients, its practical impact on individual decision outcomes is extremely limited in the current model configuration.

## 6.3 Feature Removal Experiment (Remove Gender)

A practical mitigation strategy is to remove gender from the feature set and retrain the model.

The agent compares:
- overall performance metrics (accuracy, precision, recall, F1, ROC-AUC),
- and later group-wise outcomes

If performance remains stable, removing gender can reduce discrimination risk without harming predictive quality.

In [166]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define feature lists (reuse your existing lists if already defined)
categorical_features = ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']
numerical_features = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
                      'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
                      'credit_score']

categorical_no_gender = [c for c in categorical_features if c != SENSITIVE_COL]

X_full = df.drop(TARGET_COL, axis=1)
y_full = df[TARGET_COL]

X_train_ng, X_test_ng, y_train_ng, y_test_ng = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

preprocessor_ng = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_no_gender)
    ]
)

model_ng = Pipeline(steps=[
    ("preprocessor", preprocessor_ng),
    ("classifier", LogisticRegression(max_iter=1000))
])

model_ng.fit(X_train_ng, y_train_ng)

y_pred_ng = model_ng.predict(X_test_ng)
y_proba_ng = model_ng.predict_proba(X_test_ng)[:, 1]

metrics_baseline = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

metrics_no_gender = {
    "accuracy": accuracy_score(y_test_ng, y_pred_ng),
    "precision": precision_score(y_test_ng, y_pred_ng, zero_division=0),
    "recall": recall_score(y_test_ng, y_pred_ng, zero_division=0),
    "f1": f1_score(y_test_ng, y_pred_ng, zero_division=0),
    "roc_auc": roc_auc_score(y_test_ng, y_proba_ng),
}

print("Baseline model (with gender):")
for k, v in metrics_baseline.items():
    print(f"{k:10s}: {v:.4f}")

print("\nModel without gender:")
for k, v in metrics_no_gender.items():
    print(f"{k:10s}: {v:.4f}")

Baseline model (with gender):
accuracy  : 0.8993
precision : 0.7888
recall    : 0.7470
f1        : 0.7673
roc_auc   : 0.9562

Model without gender:
accuracy  : 0.8993
precision : 0.7885
recall    : 0.7475
f1        : 0.7675
roc_auc   : 0.9562


### Feature Removal Experiment Interpretation

The removal of the gender feature does not affect model performance.

Accuracy and ROC-AUC remain identical (0.8993 and 0.9562, respectively).  
Precision, recall, and F1-score differ only at the fourth decimal place, indicating negligible variation.

These results suggest that gender does not provide unique predictive value beyond other features in the dataset.

Importantly, the model maintains strong discriminatory performance without relying on gender information.

From a responsible AI perspective, this indicates that gender can be safely removed without compromising predictive quality, thereby reducing potential bias and regulatory risk.

This strengthens the case for excluding sensitive demographic attributes in practical deployment.

## 6.4 Fairness Metrics

To quantify group disparity, the agent computes:

- **Statistical Parity Difference**:  
  difference in predicted approval rates across groups.

- **Equal Opportunity Difference**:  
  difference in true positive rates (TPR/Recall) across groups.

Values close to 0 indicate similar model behavior across groups.

In [167]:
# Expect group_report to exist from step (1)
# If your group labels differ, adjust here:
g1 = "female"
g2 = "male"

row_f = group_report[group_report["group"] == g1].iloc[0]
row_m = group_report[group_report["group"] == g2].iloc[0]

stat_parity_diff = row_f["predicted_positive_rate"] - row_m["predicted_positive_rate"]
eq_opp_diff = row_f["TPR"] - row_m["TPR"]
fpr_diff = row_f["FPR"] - row_m["FPR"]

print("Fairness metrics (female - male):")
print(f"Statistical parity difference: {stat_parity_diff:.4f}")
print(f"Equal opportunity difference (TPR): {eq_opp_diff:.4f}")
print(f"FPR difference: {fpr_diff:.4f}")

Fairness metrics (female - male):
Statistical parity difference: 0.0013
Equal opportunity difference (TPR): -0.0142
FPR difference: 0.0147


### Fairness Metrics Interpretation

The statistical parity difference between gender groups is 0.0013, indicating that predicted approval rates differ by only 0.13%. This gap is practically negligible and suggests that the model does not introduce aggregate-level approval disparities.

The equal opportunity difference (TPR gap) is −0.0142, meaning that recall for female applicants is approximately 1.4% lower than for male applicants. This difference is small and does not indicate substantial imbalance in true positive detection.

The false positive rate difference is 0.0147, indicating that female applicants experience slightly higher false positive rates by approximately 1.5%. While measurable, this gap remains modest in magnitude.

Overall, fairness metrics suggest no strong evidence of systematic gender-based discrimination in the current model configuration. Observed differences are minor and remain within a narrow range.

## Responsible AI & Fairness Audit Summary

A comprehensive fairness audit was conducted across four dimensions: group-wise performance evaluation, counterfactual sensitivity testing, feature removal experimentation, and quantitative fairness metrics.

Although gender appeared among influential model coefficients, further analysis revealed minimal practical impact.

- Predicted approval rates are nearly identical across gender groups.
- Counterfactual testing shows that flipping gender changes decisions in only 0.17% of cases.
- Removing gender does not affect model performance (ROC-AUC remains 0.9562).
- Fairness metrics indicate only minor gaps (≈1–1.5%) across groups.

Taken together, these results suggest that the model does not exhibit systematic gender-based discrimination in its current configuration.

Importantly, since predictive performance remains stable after removing gender, the model can be deployed without this sensitive attribute, reducing regulatory and ethical risk without sacrificing accuracy.

This demonstrates how analytical agents can integrate performance optimization with responsible AI principles.

## 7. Automated Agent Report

To simulate decision-support behavior, the agent aggregates:

- model performance metrics,
- feature importance analysis,
- fairness audit results,
- and mitigation conclusions

into a structured executive summary.

### Why the Agent Aggregates These Components

A decision-support agent should not function as a simple prediction engine.  
Instead, it must integrate performance evaluation, interpretability, and governance considerations into a unified reasoning process.

The automated report aggregates four essential components:

#### 1. Model Performance Metrics

Performance metrics (ROC-AUC, F1-score, precision, recall) quantify predictive quality.  
They answer the question:

> How well does the model distinguish approved from rejected applications?

Without this layer, the agent cannot justify reliability.

---

#### 2. Feature Importance Analysis

Interpretability analysis explains *why* decisions are made.

By ranking coefficients, the agent identifies dominant drivers such as:
- prior default history,
- loan amount,
- interest rate.

This transforms the system from a black-box predictor into an interpretable decision-support tool.

---

#### 3. Fairness Audit Results

Performance alone is insufficient in financial decision systems.

The agent therefore evaluates:
- group-level disparities,
- counterfactual sensitivity,
- statistical parity and equal opportunity gaps.

This ensures the system does not introduce systematic bias.

---

#### 4. Mitigation and Deployment Conclusions

A responsible agent must not only detect potential risks but also evaluate mitigation strategies.

By retraining the model without gender and observing stable performance, the agent demonstrates:

- performance robustness,
- reduced ethical risk,
- improved regulatory compliance.

---

### Conceptual Rationale

The aggregation of these components reflects a broader design principle:

> Intelligent agents should combine predictive optimization with transparency, accountability, and governance awareness.

This transforms the system from a classification model into a responsible analytical decision-support agent.

In [168]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import confusion_matrix

def _fmt(x, nd=4):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "NA"
    return f"{x:.{nd}f}"

def _confusion_summary(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    total = tn + fp + fn + tp
    tpr = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    tnr = tn / (tn + fp) if (tn + fp) else np.nan
    fnr = fn / (fn + tp) if (fn + tp) else np.nan
    return {
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp), "Total": int(total),
        "TPR": tpr, "FPR": fpr, "TNR": tnr, "FNR": fnr
    }

def _df_to_md_table(df, max_rows=10):
    # Small markdown table without heavy formatting
    d = df.copy()
    if len(d) > max_rows:
        d = d.head(max_rows)
    return d.to_markdown(index=False)

def build_rich_agent_report(
    df,
    target_col,
    metrics_baseline,
    metrics_no_gender,
    y_test, y_pred, y_proba,
    feature_importance,
    group_report,
    stat_parity_diff, eq_opp_diff, fpr_diff,
    delta, share_changed,
    threshold=0.5
) -> str:
    # -------- Data snapshot --------
    n_rows, n_cols = df.shape
    n_features = n_cols - 1
    pos_share = float(df[target_col].mean())  # assuming 1 is positive class
    neg_share = 1.0 - pos_share
    missing_total = int(df.isnull().sum().sum())
    dup_total = int(df.duplicated().sum())

    # -------- Confusion matrix summary --------
    conf = _confusion_summary(y_test, y_pred)

    # -------- Counterfactual distribution summary --------
    delta = np.asarray(delta, dtype=float)
    delta_abs = np.abs(delta)
    cf_stats = {
        "mean": float(np.mean(delta)),
        "median": float(np.median(delta)),
        "p95_abs": float(np.percentile(delta_abs, 95)),
        "max_abs": float(np.max(delta_abs)),
        "share_abs_gt_0_01": float((delta_abs > 0.01).mean()),
        "share_abs_gt_0_05": float((delta_abs > 0.05).mean()),
        "share_changed": float(share_changed)
    }

    # -------- Top drivers --------
    top_neg = feature_importance.sort_values("coefficient", ascending=True).head(7)[["feature","coefficient"]]
    top_pos = feature_importance.sort_values("coefficient", ascending=False).head(5)[["feature","coefficient"]]

    # -------- Mitigation deltas --------
    metric_keys = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    deltas = {k: float(metrics_no_gender[k] - metrics_baseline[k]) for k in metric_keys if k in metrics_baseline and k in metrics_no_gender}

    # -------- Group report (compact) --------
    grp_cols = ["group", "n", "true_positive_share", "predicted_positive_rate",
                "precision", "recall", "f1", "roc_auc", "TPR", "FPR"]
    grp_table = group_report[grp_cols].copy() if all(c in group_report.columns for c in grp_cols) else group_report.copy()

    # -------- Build markdown --------
    md = f"""
## 7. Automated Agent Report (Rich)

### Data Snapshot
- **Rows / Features:** {n_rows} / {n_features} (target: `{target_col}`)
- **Class balance (0 vs 1):** {neg_share:.3f} vs {pos_share:.3f}
- **Missing values:** {missing_total}
- **Duplicate rows:** {dup_total}

---

### Baseline Model Performance (Logistic Regression)
- **Accuracy:** {_fmt(metrics_baseline['accuracy'])}
- **Precision / Recall:** {_fmt(metrics_baseline['precision'])} / {_fmt(metrics_baseline['recall'])}
- **F1-score:** {_fmt(metrics_baseline['f1'])}
- **ROC-AUC:** {_fmt(metrics_baseline['roc_auc'])}

**Confusion matrix summary (threshold={threshold}):**
- TN={conf['TN']}, FP={conf['FP']}, FN={conf['FN']}, TP={conf['TP']}
- TPR (Recall)={_fmt(conf['TPR'])}, FPR={_fmt(conf['FPR'])}, TNR={_fmt(conf['TNR'])}, FNR={_fmt(conf['FNR'])}

---

### Key Decision Drivers (Top Coefficients)
**Strong negative factors (reduce approval probability):**
{_df_to_md_table(top_neg.assign(coefficient=top_neg["coefficient"].map(lambda x: f"{x:+.2f}")), max_rows=7)}

**Strong positive factors (increase approval probability):**
{_df_to_md_table(top_pos.assign(coefficient=top_pos["coefficient"].map(lambda x: f"{x:+.2f}")), max_rows=5)}

---

### Fairness Audit (Gender)
**Group-wise performance snapshot:**
{_df_to_md_table(grp_table, max_rows=10)}

**Group-level differences (female − male):**
- Statistical parity difference: {stat_parity_diff:+.4f}
- Equal opportunity difference (TPR gap): {eq_opp_diff:+.4f}
- FPR difference: {fpr_diff:+.4f}

**Counterfactual sensitivity (flip gender only):**
- Mean Δprobability: {cf_stats['mean']:+.4f}
- Median Δprobability: {cf_stats['median']:+.4f}
- 95th percentile |Δprobability|: {_fmt(cf_stats['p95_abs'])}
- Max |Δprobability|: {_fmt(cf_stats['max_abs'])}
- Share with |Δ| > 0.01: {_fmt(cf_stats['share_abs_gt_0_01'])}
- Share with |Δ| > 0.05: {_fmt(cf_stats['share_abs_gt_0_05'])}
- Share of changed decisions (thr={threshold}): {_fmt(cf_stats['share_changed'])}

---

### Mitigation Check: Remove Gender
**Model without gender:**
- Accuracy: {_fmt(metrics_no_gender['accuracy'])}
- Precision / Recall: {_fmt(metrics_no_gender['precision'])} / {_fmt(metrics_no_gender['recall'])}
- F1-score: {_fmt(metrics_no_gender['f1'])}
- ROC-AUC: {_fmt(metrics_no_gender['roc_auc'])}

**Metric deltas (no-gender − baseline):**
{_df_to_md_table(pd.DataFrame([{"metric": k, "delta": f"{deltas[k]:+.4f}"} for k in deltas]), max_rows=10)}

---

### Recommendation (Data-driven)
- **Primary:** Exclude gender from the feature set, since performance remains stable while reducing ethical and regulatory risk.
- **Monitoring:** Keep group-wise error rates (TPR/FPR) under periodic review and re-audit after dataset or policy changes.

This report demonstrates an analytical decision-support agent that integrates:
**predictive modeling → interpretability → fairness evaluation → mitigation validation**.
"""
    return md


# ---- Build & display the report ----
report_md = build_rich_agent_report(
    df=df,
    target_col=TARGET_COL,
    metrics_baseline=metrics_baseline,
    metrics_no_gender=metrics_no_gender,
    y_test=y_test, y_pred=y_pred, y_proba=y_proba,
    feature_importance=feature_importance,
    group_report=group_report,
    stat_parity_diff=stat_parity_diff,
    eq_opp_diff=eq_opp_diff,
    fpr_diff=fpr_diff,
    delta=delta,
    share_changed=share_changed,
    threshold=0.5
)

display(Markdown(report_md))


## 7. Automated Agent Report (Rich)

### Data Snapshot
- **Rows / Features:** 45000 / 13 (target: `loan_status`)
- **Class balance (0 vs 1):** 0.778 vs 0.222
- **Missing values:** 0
- **Duplicate rows:** 0

---

### Baseline Model Performance (Logistic Regression)
- **Accuracy:** 0.8993
- **Precision / Recall:** 0.7888 / 0.7470
- **F1-score:** 0.7673
- **ROC-AUC:** 0.9562

**Confusion matrix summary (threshold=0.5):**
- TN=6600, FP=400, FN=506, TP=1494
- TPR (Recall)=0.7470, FPR=0.0571, TNR=0.9429, FNR=0.2530

---

### Key Decision Drivers (Top Coefficients)
**Strong negative factors (reduce approval probability):**
| feature                            |   coefficient |
|:-----------------------------------|--------------:|
| previous_loan_defaults_on_file_Yes |         -5.07 |
| person_home_ownership_OWN          |         -1.82 |
| loan_intent_VENTURE                |         -1    |
| person_gender_female               |         -0.98 |
| person_gender_male                 |         -0.96 |
| loan_intent_EDUCATION              |         -0.7  |
| loan_amnt                          |         -0.61 |

**Strong positive factors (increase approval probability):**
| feature                           |   coefficient |
|:----------------------------------|--------------:|
| previous_loan_defaults_on_file_No |          3.14 |
| loan_percent_income               |          1.35 |
| loan_int_rate                     |          0.99 |
| person_home_ownership_RENT        |          0.31 |
| loan_intent_DEBTCONSOLIDATION     |          0.18 |

---

### Fairness Audit (Gender)
**Group-wise performance snapshot:**
| group   |    n |   true_positive_share |   predicted_positive_rate |   precision |   recall |     f1 |   roc_auc |    TPR |    FPR |
|:--------|-----:|----------------------:|--------------------------:|------------:|---------:|-------:|----------:|-------:|-------:|
| female  | 3945 |                0.2165 |                    0.2112 |      0.7575 |   0.7389 | 0.7481 |    0.9522 | 0.7389 | 0.0654 |
| male    | 5055 |                0.2267 |                    0.2099 |      0.8134 |   0.7531 | 0.7821 |    0.9594 | 0.7531 | 0.0507 |

**Group-level differences (female − male):**
- Statistical parity difference: +0.0013
- Equal opportunity difference (TPR gap): -0.0142
- FPR difference: +0.0147

**Counterfactual sensitivity (flip gender only):**
- Mean Δprobability: -0.0002
- Median Δprobability: -0.0000
- 95th percentile |Δprobability|: 0.0046
- Max |Δprobability|: 0.0047
- Share with |Δ| > 0.01: 0.0000
- Share with |Δ| > 0.05: 0.0000
- Share of changed decisions (thr=0.5): 0.0017

---

### Mitigation Check: Remove Gender
**Model without gender:**
- Accuracy: 0.8993
- Precision / Recall: 0.7885 / 0.7475
- F1-score: 0.7675
- ROC-AUC: 0.9562

**Metric deltas (no-gender − baseline):**
| metric    |   delta |
|:----------|--------:|
| accuracy  |  0      |
| precision | -0.0003 |
| recall    |  0.0005 |
| f1        |  0.0001 |
| roc_auc   | -0      |

---

### Recommendation (Data-driven)
- **Primary:** Exclude gender from the feature set, since performance remains stable while reducing ethical and regulatory risk.
- **Monitoring:** Keep group-wise error rates (TPR/FPR) under periodic review and re-audit after dataset or policy changes.

This report demonstrates an analytical decision-support agent that integrates:
**predictive modeling → interpretability → fairness evaluation → mitigation validation**.
